# ViT数据融合过程演示


代码参考链接：

[modeling_kimi_k25.py](https://huggingface.co/moonshotai/Kimi-K2.5/blob/main/modeling_kimi_k25.py)


Author: kaiyuan

Email: kaiyuanxie@yeah.net

## 1 视觉数据处理依赖函数导入

In [ ]:
# meida utils从模型文件中导入
# 地址：https://huggingface.co/moonshotai/Kimi-K2.5/blob/main/media_utils.py
import base64
import io
import math
import os
from datetime import datetime, timezone
from typing import List, Literal, Optional, TypedDict

import numpy as np
from PIL import Image
from pydantic import BaseModel, Field

try:
    from mecord import VideoReader
except ImportError:
    VideoReader = None


class VideoSpec(BaseModel):
    media_type: str = Literal['video']
    height: int = Field(..., gt=0, description="video frame height")
    width: int = Field(..., gt=0, description="video frame width")
    num_frames: int = Field(..., gt=0, description="num frames")
    fps: float = Field(..., gt=0, description="average fps")

    # optional, help to accelerate video reading
    key_indices: list[int] = Field(None, description="key indices")
    frame_time_info: dict = Field(None, description="frame time info")


class ImageInput(TypedDict):
    type: Literal['image']
    image: Image.Image


class VideoChunkInput(TypedDict):
    type: Literal['video_chunk']
    video_chunk: List[Image.Image]
    prompt: Optional[str] = None


MediaInput = ImageInput | VideoChunkInput


def get_video_meta(video_src: bytes | str | os.PathLike,
                   accurate: bool = True) -> dict:
    """Get the dimensions of a video."""
    if isinstance(video_src, os.PathLike):
        video_src = str(video_src)
    # if b64 string, decode to bytes
    if isinstance(video_src,
                  str) and video_src.startswith('data:video/mp4;base64,'):
        video_src = base64.b64decode(video_src.split(',')[1])
    video = VideoReader(video_src, auto_init=accurate, num_threads=1)
    assert video.num_frames > 0, "Invalid video format."
    assert video.original_width > 0 and video.original_height > 0, (
        "Invalid video format.")
    assert video.avg_fps > 0, "Invalid video format."
    return VideoSpec(media_type='video',
                     height=video.original_height,
                     width=video.original_width,
                     num_frames=video.num_frames,
                     fps=video.avg_fps,
                     key_indices=video.key_indices,
                     frame_time_info=video.frame_time_info)


def timestamp_as_str(timestamp: float,
                     timestamp_mode: str = "hh:mm:ss.fff") -> str:
    """Convert a timestamp to a string in the format of HH:MM:SS.mmm."""
    if timestamp_mode == "hh:mm:ss.fff":
        return (datetime.fromtimestamp(timestamp,
                                       tz=timezone.utc).strftime("%H:%M:%S") +
                f".{int((timestamp % 1) * 1000):03d}")
    elif timestamp_mode == "mm:ss.fff":
        return (datetime.fromtimestamp(timestamp,
                                       tz=timezone.utc).strftime("%M:%S") +
                f".{int((timestamp % 1) * 1000):03d}")
    elif timestamp_mode == "mm:ss":
        return datetime.fromtimestamp(timestamp,
                                      tz=timezone.utc).strftime("%M:%S")
    else:
        raise ValueError(f"Invalid timestamp mode: {timestamp_mode}")


def navit_resize_image(
    width: int,
    height: int,
    patch_size: int,
    merge_kernel_size: int,
    in_patch_limit: int,
    patch_limit_on_one_side: int,
    fixed_output_tokens: int | None,
):
    # Apply the patch limits.
    s1 = math.sqrt(
        in_patch_limit /
        (max(1.0, width // patch_size) * max(1.0, height // patch_size)))
    s2 = patch_limit_on_one_side * patch_size / width
    s3 = patch_limit_on_one_side * patch_size / height
    scale = min(1.0, s1, s2, s3)
    new_w, new_h = max(1, int(width * scale)), max(1, int(height * scale))
    new_w = min(new_w, patch_limit_on_one_side * patch_size)
    new_h = min(new_h, patch_limit_on_one_side * patch_size)

    # Calculate the padding to make the height and width divisible by the merge kernel size and patch size.
    factor = merge_kernel_size * patch_size

    pad_height = (factor - new_h % factor) % factor
    pad_width = (factor - new_w % factor) % factor

    if fixed_output_tokens is not None:
        num_tokens = fixed_output_tokens
    else:
        # Calculate new dimensions after padding and patching
        token_height = (new_h + pad_height) // factor
        token_width = (new_w + pad_width) // factor

        assert token_height * merge_kernel_size <= patch_limit_on_one_side, (
            f"token_height {token_height} * merge_kernel_size {merge_kernel_size} > patch_limit_on_one_side {patch_limit_on_one_side}"
        )
        assert token_width * merge_kernel_size <= patch_limit_on_one_side, (
            f"token_width {token_width} * merge_kernel_size {merge_kernel_size} > patch_limit_on_one_side {patch_limit_on_one_side}"
        )

        num_tokens = token_height * token_width
    return {
        "num_tokens": num_tokens,
        "new_width": new_w,
        "new_height": new_h,
        "pad_width": pad_width,
        "pad_height": pad_height,
        "sampled_nframes": 1,
    }


def navit_resize_video(
    width: int,
    height: int,
    nframes: int,
    avg_fps: float,
    sample_fps: float,
    patch_size: int,
    merge_kernel_size: int,
    in_patch_limit_each_frame: int,
    patch_limit_on_one_side: int,
    in_patch_limit_total: int | None,
    max_num_frames_each_video: int | None,
    fixed_output_tokens_each_frame: int | None,
):
    sample_fps = min(sample_fps, avg_fps)
    # Calculate the number of frames to sample based on target FPS
    sampled_nframes = max(round(nframes * sample_fps / avg_fps), 1)
    if max_num_frames_each_video is not None:
        sampled_nframes = min(sampled_nframes, max_num_frames_each_video)

    if in_patch_limit_total is not None:
        in_patch_limit_each_frame = min(
            round(in_patch_limit_total / sampled_nframes),
            in_patch_limit_each_frame)

    ret = navit_resize_image(
        width,
        height,
        patch_size,
        merge_kernel_size,
        in_patch_limit_each_frame,
        patch_limit_on_one_side,
        fixed_output_tokens_each_frame,
    )
    ret["sampled_nframes"] = sampled_nframes
    return ret


def real_sample_fps_and_max_num_frames(
    type_name: Literal["video", "video_chunk"],
    sample_fps: float,
    max_num_frames_each_video: int | None,
) -> tuple[int, int | None]:
    if type_name == "video":
        return sample_fps, max_num_frames_each_video
    elif type_name == "video_chunk":
        max_num_frames_each_video = None
        sample_fps = math.inf
        return sample_fps, max_num_frames_each_video
    else:
        return math.inf, None


def _to_pil(data: str | bytes):
    if isinstance(data, Image.Image):

        return data.convert("RGB")
    elif isinstance(data, str):
        if data.startswith("data:"):
            raw_base64 = data.split(",")[1]
            return Image.open(io.BytesIO(
                base64.b64decode(raw_base64))).convert("RGB")
        else:
            return Image.open(data).convert("RGB")
    elif isinstance(data, bytes):
        return Image.open(io.BytesIO(data)).convert("RGB")
    else:
        raise ValueError(f"Unsupported data type: {type(data)}")


def ensure_media_type(media: MediaInput) -> MediaInput:
    if media['type'] == 'image':
        media['image'] = _to_pil(media['image'])
        return media
    elif media['type'] == 'video_chunk':
        media['video_chunk'] = [
            _to_pil(frame) for frame in media['video_chunk']
        ]
        return media
    else:
        raise ValueError(f"Unsupported media type: {media['type']}")


def image_to_np(
    image: Image.Image,
    resize_to: tuple[int, int] | None = None,
    mode: str = "resize",
    raise_error_for_ill_resize: bool = True,
) -> np.ndarray:
    """Convert an image to a numpy array.
    Args:
        content: The image to convert.
        resize_to: The size to resize the image to.
        mode: The mode to resize the image to.
        raise_error_for_ill_resize: Whether to raise an error for ill-sized resize.
    Returns:
        A numpy array.
    """
    assert isinstance(image, Image.Image), "image must be a PIL Image"
    if resize_to is not None:
        if mode == "resize":
            image = image.resize(resize_to, resample=Image.Resampling.BICUBIC)

        elif mode == "rescale_and_pad_to_center":
            scale = min(resize_to[0] / image.width,
                        resize_to[1] / image.height, 1.0)
            new_width = round(image.width * scale)
            new_height = round(image.height * scale)
            if new_width == 0 or new_height == 0:
                if raise_error_for_ill_resize:
                    raise ValueError(
                        f"Invalid resize to: {resize_to}, from image size: {image.size}"
                    )
                else:
                    return np.zeros((resize_to[1], resize_to[0], 3),
                                    dtype=np.uint8)

            image = image.resize((new_width, new_height),
                                 resample=Image.Resampling.BICUBIC)
            padding_left = (resize_to[0] - new_width) // 2
            padding_right = resize_to[0] - new_width - padding_left
            padding_top = (resize_to[1] - new_height) // 2
            padding_bottom = resize_to[1] - new_height - padding_top
            image = np.asarray(image)
            image = np.pad(
                image,
                ((padding_top, padding_bottom), (padding_left, padding_right),
                 (0, 0)),
                mode="constant",
                constant_values=0,
            )
            assert image.shape == (resize_to[1], resize_to[0], 3)

        elif mode == "rescale_and_pad_to_rightbottom":
            scale = min(resize_to[0] / image.width,
                        resize_to[1] / image.height, 1.0)
            new_width = round(image.width * scale)
            new_height = round(image.height * scale)
            if new_width == 0 or new_height == 0:
                if raise_error_for_ill_resize:
                    raise ValueError(
                        f"Invalid resize to: {resize_to}, from image size: {image.size}"
                    )
                else:
                    return np.zeros((resize_to[1], resize_to[0], 3),
                                    dtype=np.uint8)

            image = image.resize((new_width, new_height),
                                 resample=Image.Resampling.BICUBIC)
            padding_right = resize_to[0] - new_width
            padding_bottom = resize_to[1] - new_height
            image = np.asarray(image)
            image = np.pad(
                image,
                ((0, padding_bottom), (0, padding_right), (0, 0)),
                mode="constant",
                constant_values=0,
            )
            assert image.shape == (resize_to[1], resize_to[0], 3)

        else:
            raise ValueError(f"Invalid mode: {mode}")

    if isinstance(image, Image.Image):
        return np.asarray(image)
    else:
        return image


def navit_patchify(pixel_values: np.ndarray,
                   patch_size: int) -> dict[str, np.ndarray]:
    """Reshape the pixel values to a navit shape.
    Args:
        pixel_values: np.ndarray, shape (t, h, w, c)
        patch_size: int
    Returns:
        dict[str, np.ndarray]
        - patches: np.ndarray, shape (t * h//patch_size * w//patch_size, c, patch_size, patch_size)
        - grid_thw: np.ndarray, (t, h//patch_size, w//patch_size)
    """
    T, H, W, C = pixel_values.shape
    assert C == 3, "pixel_values must have 3 channels"

    patches = pixel_values.reshape(T, H // patch_size, patch_size,
                                   W // patch_size, patch_size, C)
    # (T, H//patch_size, W//patch_size, C, patch_size, patch_size)
    patches = patches.transpose(0, 1, 3, 5, 2, 4)
    patches = patches.reshape(-1, C, patch_size, patch_size)
    grid_thw = np.array([T, H // patch_size, W // patch_size])
    return {"pixel_values": patches, "grid_thw": grid_thw}


def normalize(x: np.ndarray,
              mean,
              std_inv,
              pixels_dtype: np.dtype = np.float32) -> np.ndarray:
    """Normalize the image.
    Args:
        x: The image to normalize. The shape is (..., 3). The dtype is uint8. The range is [0, 255].
        mean: The mean of the image.
        std_inv: The inverse of the std of the image.
        pixels_dtype: The dtype of the image.
    Returns:
        The normalized image. The shape is (..., 3). The dtype is determined by the pixels_dtype.
    """
    x = (x / 255.0).astype(pixels_dtype)
    x -= mean
    x *= std_inv
    return x


def _to_tensor(data, **kwargs):
    import torch

    if isinstance(data, np.ndarray):
        return torch.from_numpy(data).to(**kwargs)
    elif isinstance(data, torch.Tensor):
        return data.to(**kwargs)
    elif isinstance(data, list):
        return [_to_tensor(item, **kwargs) for item in data]
    elif isinstance(data, tuple):
        return tuple(_to_tensor(item, **kwargs) for item in data)
    elif isinstance(data, dict):
        return {k: _to_tensor(v, **kwargs) for k, v in data.items()}
    elif data is None:
        return None
    else:
        raise ValueError(f"Unsupported data type: {type(data)}")


## 2 处理过程示意代码

In [ ]:
import argparse
import json
from pathlib import Path

import numpy as np
import torch
from PIL import Image


DEFAULT_MEDIA_PROC_CFG = {
    "in_patch_limit": 16384,
    "patch_size": 14,
    "image_mean": [0.5, 0.5, 0.5],
    "image_std": [0.5, 0.5, 0.5],
    "merge_kernel_size": 2,
    "fixed_output_tokens": None,
    "patch_limit_on_one_side": 512,
    "in_patch_limit_each_frame": 4096,
    "in_patch_limit_video": None,
    "sample_fps": 2.0,
    "max_num_frames_each_video": None,
    "temporal_merge_kernel_size": 4,
    "timestamp_mode": "hh:mm:ss.fff",
}

# Define a default model configuration for Colab environment since config.json might be missing.
# Values are illustrative and chosen to allow the demo to run without errors.
DEFAULT_MODEL_CFG = {
    "vision_config": {
        "vt_hidden_size": 1024, # Example value
        "merge_kernel_size": (2, 2), # Expected to be a tuple for kh, kw unpacking
        "text_hidden_size": 768, # Example value
    },
    "media_placeholder_token_id": 32000, # Example value
    "text_config": {
        "vocab_size": 32001, # Example value
    },
}


def _read_json_file(path: Path) -> dict:
    # 使用 utf-8-sig 兼容带 BOM 的 JSON 文件
    raw = path.read_text(encoding="utf-8-sig").strip()
    if not raw:
        raise ValueError(f"{path.name} 为空文件")
    obj = json.loads(raw)
    if not isinstance(obj, dict):
        raise ValueError(f"{path.name} 顶层不是 JSON object")
    return obj


def load_configs(base_dir: Path) -> tuple[dict, dict]:
    model_cfg = None
    model_cfg_path = base_dir / "config.json"
    try:
        model_cfg = _read_json_file(model_cfg_path)
    except Exception as exc:
        print(
            f"[warn] 读取 {model_cfg_path.name} 失败: {exc}\n"
            "[warn] 将回退到内置默认 model_cfg 继续演示。"
        )
        model_cfg = DEFAULT_MODEL_CFG.copy()

    media_proc_cfg = None
    pre_cfg_path = base_dir / "preprocessor_config.json"
    try:
        pre_cfg = _read_json_file(pre_cfg_path)
        media_proc_cfg = pre_cfg.get("media_proc_cfg")
        if not isinstance(media_proc_cfg, dict):
            raise ValueError("缺少 media_proc_cfg 字段")
    except Exception as exc:
        print(
            f"[warn] 读取 {pre_cfg_path.name} 失败: {exc}\n"
            "[warn] 将回退到内置默认 media_proc_cfg 继续演示。"
        )
        media_proc_cfg = DEFAULT_MEDIA_PROC_CFG.copy()

    return media_proc_cfg, model_cfg


def load_image(image_path: str | None, width: int = 640, height: int = 480) -> Image.Image:
    if image_path is None:
        arr = np.random.randint(0, 256, size=(height, width, 3), dtype=np.uint8)
        return Image.fromarray(arr, mode="RGB")
    return Image.open(image_path).convert("RGB")


def demo_image_to_merge(image: Image.Image, media_proc_cfg: dict, model_cfg: dict) -> None:
    patch_size = media_proc_cfg["patch_size"]
    merge_kernel_size = media_proc_cfg["merge_kernel_size"]
    vt_cfg = model_cfg["vision_config"]

    print("=== 1) 输入图片 ===")
    print(f"PIL size (W,H): {image.size}")

    resize_cfg = navit_resize_image(
        width=image.width,
        height=image.height,
        patch_size=patch_size,
        merge_kernel_size=merge_kernel_size,
        in_patch_limit=media_proc_cfg["in_patch_limit"],
        patch_limit_on_one_side=media_proc_cfg["patch_limit_on_one_side"],
        fixed_output_tokens=media_proc_cfg["fixed_output_tokens"],
    )
    print("\n=== 2) resize/pad 配置 ===")
    print(resize_cfg)

    image_np = image_to_np(
        image,
        resize_to=(resize_cfg["new_width"], resize_cfg["new_height"]),
        mode="resize",
    )
    image_np = np.pad(
        image_np,
        ((0, resize_cfg["pad_height"]), (0, resize_cfg["pad_width"]), (0, 0)),
        mode="constant",
        constant_values=0,
    )
    image_np_t = np.expand_dims(image_np, axis=0)
    print("\n=== 3) resize + pad 后 ===")
    print(f"pixel_values (T,H,W,C): {image_np_t.shape}")

    image_std_inv = 1.0 / np.array(media_proc_cfg["image_std"])
    image_mean = np.array(media_proc_cfg["image_mean"])
    normalized = normalize(image_np_t, image_mean, image_std_inv)
    print("\n=== 4) normalize 后 ===")
    print(f"normalized (T,H,W,C): {normalized.shape}")

    patchified = navit_patchify(normalized, patch_size=patch_size)
    pixel_values = torch.from_numpy(patchified["pixel_values"])
    grid_thw = torch.from_numpy(patchified["grid_thw"]).to(torch.int64)
    grid_thws = grid_thw.unsqueeze(0)
    print("\n=== 5) patchify 后 ===")
    print(f"pixel_values (L,C,P,P): {tuple(pixel_values.shape)}")
    print(f"grid_thw (T,H',W'): {tuple(grid_thw.tolist())}")
    print(f"grid_thws (B,3): {tuple(grid_thws.shape)}")

    # 对应 MoonVision3dPatchEmbed.proj 后，token 维度从 C*P*P -> vt_hidden_size。
    vt_hidden = vt_cfg["vt_hidden_size"]
    t, h, w = grid_thw.tolist()
    tokens_before_merge = t * h * w
    vision_hidden = torch.randn(tokens_before_merge, vt_hidden, dtype=torch.float32)
    print("\n=== 6) vision tower 编码后(示意) ===")
    print(f"vision_hidden (sum(T*H'*W'), vt_hidden): {tuple(vision_hidden.shape)}")

    # 对应 tpool_patch_merger: 按 merge_kernel_size 做空间降采样 + 时间池化。
    kh, kw = vt_cfg["merge_kernel_size"]
    new_h, new_w = h // kh, w // kw
    merged = vision_hidden.view(t, new_h, kh, new_w, kw, vt_hidden).permute(0, 1, 3, 2, 4, 5).contiguous().mean(dim=0)
    merged = merged.view(new_h * new_w, kh * kw, vt_hidden)
    print("\n=== 7) tpool_patch_merger 后 ===")
    print(f"merged_for_projector (N_img_tokens, K, vt_hidden): {tuple(merged.shape)}")

    # 对应 PatchMergerMLP: LayerNorm(mm_hidden_size) + flatten(K*mm_hidden_size) + MLP -> text_hidden_size
    text_hidden = vt_cfg["text_hidden_size"]
    projected = torch.randn(merged.shape[0], text_hidden, dtype=torch.float32)
    print("\n=== 8) mm_projector 后(示意) ===")
    print(f"image_features (N_img_tokens, text_hidden): {tuple(projected.shape)}")

    # 对应 _merge_input_ids_with_image_features 之前:
    # 文本 embedding shape 通常是 (B, seq_len, text_hidden)
    B, seq_len = 1, 12
    placeholder_id = model_cfg["media_placeholder_token_id"]
    vocab_size = int(model_cfg["text_config"]["vocab_size"])
    input_ids = torch.randint(low=0, high=min(vocab_size, 5000), size=(B, seq_len), dtype=torch.long)
    input_ids[0, 3] = placeholder_id
    attention_mask = torch.ones((B, seq_len), dtype=torch.long)
    text_inputs_embeds = torch.randn(B, seq_len, text_hidden, dtype=torch.float32)
    print("\n=== 9) merge 前文本输入(示意) ===")
    print(f"input_ids: {tuple(input_ids.shape)}")
    print(f"inputs_embeds(text): {tuple(text_inputs_embeds.shape)}")
    print(f"attention_mask: {tuple(attention_mask.shape)}")
    print(f"placeholder token id: {placeholder_id} (位置=3)")

    # 按 modeling.py 的占位逻辑估算 merge 后长度:
    final_seq_len = seq_len - 1 + projected.shape[0]
    print("\n=== 10) 与图像特征 merge 后(进入LLM前) ===")
    print(f"merged_inputs_embeds: ({B}, {final_seq_len}, {text_hidden})")
    print(f"merged_attention_mask: ({B}, {final_seq_len})")
    print(f"说明: 1个占位符token被替换为 {projected.shape[0]} 个图像token。")


def main() -> None:
    image_path = None # 可以替换成自己的图片

    base_dir = Path('.')
    media_proc_cfg, model_cfg = load_configs(base_dir)
    image = load_image(image_path)
    demo_image_to_merge(image, media_proc_cfg, model_cfg)


if __name__ == "__main__":
    main()

[warn] 读取 config.json 失败: [Errno 2] No such file or directory: 'config.json'
[warn] 将回退到内置默认 model_cfg 继续演示。
[warn] 读取 preprocessor_config.json 失败: [Errno 2] No such file or directory: 'preprocessor_config.json'
[warn] 将回退到内置默认 media_proc_cfg 继续演示。
=== 1) 输入图片 ===
PIL size (W,H): (640, 480)

=== 2) resize/pad 配置 ===
{'num_tokens': 414, 'new_width': 640, 'new_height': 480, 'pad_width': 4, 'pad_height': 24, 'sampled_nframes': 1}

=== 3) resize + pad 后 ===
pixel_values (T,H,W,C): (1, 504, 644, 3)

=== 4) normalize 后 ===
normalized (T,H,W,C): (1, 504, 644, 3)

=== 5) patchify 后 ===
pixel_values (L,C,P,P): (1656, 3, 14, 14)
grid_thw (T,H',W'): (1, 36, 46)
grid_thws (B,3): (1, 3)

=== 6) vision tower 编码后(示意) ===
vision_hidden (sum(T*H'*W'), vt_hidden): (1656, 1024)

=== 7) tpool_patch_merger 后 ===
merged_for_projector (N_img_tokens, K, vt_hidden): (414, 4, 1024)

=== 8) mm_projector 后(示意) ===
image_features (N_img_tokens, text_hidden): (414, 768)

=== 9) merge 前文本输入(示意) ===
input_ids: (1, 12

/tmp/ipykernel_892/1351483198.py:84: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  return Image.fromarray(arr, mode="RGB")
